# Lens Research Studio

Gradio UI: scoping questions, audience lens, web search count (2–5), parallel search, optional follow-up round, report, evaluator, and one revision pass when needed.

Set `OPENAI_API_KEY` or OpenRouter env vars in `.env` at the repo root. Run cells in order; the last cell starts the app.


In [ ]:
%pip install -q openai-agents gradio python-dotenv pydantic


In [ ]:
from __future__ import annotations

import asyncio
import os

import gradio as gr
from dotenv import load_dotenv
from pydantic import BaseModel, Field, model_validator

from agents import Agent, ModelSettings, OpenAIProvider, RunConfig, Runner, WebSearchTool, gen_trace_id, trace

load_dotenv(override=True)

if not (os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY")):
    raise ValueError("Set OPENAI_API_KEY or OPENROUTER_API_KEY in .env (repo root).")


def get_run_config() -> RunConfig:
    api_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY")
    base_url = os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE")
    if not base_url:
        base_url = None
    if not base_url and os.getenv("OPENROUTER_API_KEY") and not os.getenv("OPENAI_API_KEY"):
        base_url = "https://openrouter.ai/api/v1"
    return RunConfig(model_provider=OpenAIProvider(api_key=api_key, base_url=base_url))


RC = get_run_config()
MODEL = "gpt-4o-mini"
WEB = WebSearchTool(search_context_size="low")

LENS_HINTS = {
    "Executive": "Prioritize decisions, stakeholders, ROI, and actionable takeaways.",
    "Technical": "Prioritize mechanisms, tools, APIs, architecture, and trade-offs.",
    "Risk & ethics": "Prioritize misuse, safety, regulation, equity, and governance.",
}


In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search matters for the brief.")
    query: str = Field(description="Web search query.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="Distinct web searches.")


class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        min_length=3,
        max_length=3,
        description="Three scoping questions for the user.",
    )


class FollowUpPlan(BaseModel):
    need_follow_up: bool = Field(description="True only if more web search clearly helps.")
    searches: list[WebSearchItem] = Field(default_factory=list)

    @model_validator(mode="after")
    def _cap(self):
        self.searches = self.searches[:3]
        if not self.need_follow_up:
            self.searches = []
        return self


class ReportData(BaseModel):
    short_summary: str = Field(description="2–3 sentence summary.")
    markdown_report: str = Field(description="Full markdown report.")
    follow_up_questions: list[str] = Field(description="Further research ideas.")


class EvaluationBrief(BaseModel):
    adequate: bool = Field(description="Whether the report is good enough to ship.")
    comment: str = Field(description="Short review.")
    revision_brief: str = Field(default="", description="Concrete fixes if not adequate.")


search_agent = Agent(
    name="SearchAgent",
    instructions=(
        "Search the web for the given term. Return a tight summary under 280 words: "
        "bullets or short paragraphs, facts only, no preamble."
    ),
    tools=[WEB],
    model=MODEL,
    model_settings=ModelSettings(tool_choice="required"),
)

clarify_agent = Agent(
    name="ClarifyAgent",
    instructions=(
        "Given a research topic and an audience lens, output exactly three short clarifying "
        "questions that narrow scope, timeframe, or success criteria."
    ),
    model=MODEL,
    output_type=ClarifyingQuestions,
)

expansion_agent = Agent(
    name="ExpansionAgent",
    instructions=(
        "Given the brief and existing search summaries, decide if up to three more targeted "
        "searches would materially improve coverage. Otherwise set need_follow_up false."
    ),
    model=MODEL,
    output_type=FollowUpPlan,
)

writer_agent = Agent(
    name="WriterAgent",
    instructions=(
        "Write a structured markdown report from the brief and search notes. "
        "Respect the stated lens. Do not invent citations beyond the notes."
    ),
    model=MODEL,
    output_type=ReportData,
)

evaluator_agent = Agent(
    name="EvaluatorAgent",
    instructions=(
        "Judge whether the markdown report satisfies the brief and lens. "
        "If not adequate, give a short revision_brief the writer can apply in one pass."
    ),
    model=MODEL,
    output_type=EvaluationBrief,
)


def make_planner(num_searches: int, lens: str) -> Agent:
    hint = LENS_HINTS[lens]
    return Agent(
        name="PlannerAgent",
        instructions=(
            f"Propose exactly {num_searches} distinct web searches for the brief. {hint} "
            "Each item: reason + focused query string."
        ),
        model=MODEL,
        output_type=WebSearchPlan,
    )


In [ ]:
class LensResearchStudio:
    def __init__(self) -> None:
        self.rc = RC

    async def clarify(self, topic: str, lens: str) -> str:
        topic = (topic or "").strip()
        if not topic:
            return "Enter a research topic above."
        hint = LENS_HINTS[lens]
        prompt = f"Topic:\n{topic}\n\nLens: {lens}\nGuidance: {hint}"
        result = await Runner.run(clarify_agent, prompt, run_config=self.rc)
        cq = result.final_output_as(ClarifyingQuestions)
        lines = "\n".join(f"{i}. {q}" for i, q in enumerate(cq.questions, start=1))
        return f"### Scoping questions\n\n{lines}\n\n_Answer them below, then run step 2._"

    async def _search_many(self, items: list[WebSearchItem]) -> list[str]:
        async def one(it: WebSearchItem) -> str | None:
            prompt = f"Query: {it.query}\nReason: {it.reason}"
            try:
                r = await Runner.run(search_agent, prompt, run_config=self.rc)
                return str(r.final_output)
            except Exception:
                return None

        tasks = [asyncio.create_task(one(x)) for x in items]
        out: list[str] = []
        for t in asyncio.as_completed(tasks):
            x = await t
            if x:
                out.append(x)
        return out

    async def run(self, topic: str, answers: str, lens: str, num_searches: int):
        topic = (topic or "").strip()
        if not topic:
            yield "Enter a research topic."
            return
        hint = LENS_HINTS[lens]
        brief = (
            f"Topic:\n{topic}\n\nLens: {lens}\n{hint}\n\nUser clarifications:\n"
            f"{(answers or '').strip() or '(none provided)'}\n"
        )
        trace_id = gen_trace_id()
        lines: list[str] = []

        def push(msg: str) -> str:
            lines.append(msg)
            return "\n\n".join(lines)

        with trace("Lens research studio", trace_id=trace_id):
            yield push(f"**Trace** · [OpenAI trace](https://platform.openai.com/traces/trace?trace_id={trace_id})")
            planner = make_planner(int(num_searches), lens)
            yield push("**Planning** web searches…")
            pr = await Runner.run(planner, brief, run_config=self.rc)
            plan = pr.final_output_as(WebSearchPlan)
            items = plan.searches[: int(num_searches)]
            yield push(f"**Searching** · {len(items)} queries (parallel)…")
            notes = await self._search_many(items)
            yield push("**Expansion agent** · checking for follow-up searches…")
            joined = "\n---\n".join(notes[:25])
            er = await Runner.run(
                expansion_agent,
                f"{brief}\n\nSummaries:\n{joined}",
                run_config=self.rc,
            )
            fu = er.final_output_as(FollowUpPlan)
            if fu.searches:
                yield push(f"**Follow-up** · {len(fu.searches)} extra queries…")
                notes.extend(await self._search_many(fu.searches))
            else:
                yield push("**Follow-up** · not needed.")
            yield push("**Writing** report…")
            payload = f"{brief}\n\nNotes:\n" + "\n---\n".join(notes)
            wr = await Runner.run(writer_agent, payload, run_config=self.rc)
            report = wr.final_output_as(ReportData)
            yield push("**Evaluator** · quality check…")
            evr = await Runner.run(
                evaluator_agent,
                f"Brief:\n{brief}\n\nReport:\n{report.markdown_report}",
                run_config=self.rc,
            )
            ev = evr.final_output_as(EvaluationBrief)
            yield push(f"_Review:_ {ev.comment}")
            if (not ev.adequate) and ev.revision_brief.strip():
                yield push("**Revising** from evaluator feedback…")
                payload2 = (
                    payload + "\n\nPrevious report:\n" + report.markdown_report
                    + "\n\nRevise per:\n" + ev.revision_brief
                )
                wr2 = await Runner.run(writer_agent, payload2, run_config=self.rc)
                report = wr2.final_output_as(ReportData)
            yield push("---\n\n## Report\n\n" + report.markdown_report)
            yield push(
                "\n\n### Follow-ups\n" + "\n".join(f"- {q}" for q in report.follow_up_questions)
            )


STUDIO = LensResearchStudio()


In [ ]:
async def ui_clarify(topic, lens):
    return await STUDIO.clarify(topic, lens)


async def ui_run(topic, answers, lens, depth):
    text = ""
    async for chunk in STUDIO.run(topic, answers, lens, int(depth)):
        text = chunk
        yield text


with gr.Blocks(
    theme=gr.themes.Default(primary_hue=gr.themes.colors.cyan, neutral_hue=gr.themes.colors.slate),
    title="Lens Research Studio",
) as demo:
    gr.Markdown(
        "## Lens Research Studio\n"
        "Pick **who** this is for, **how deep** to search, then **scope** → **run**."
    )
    with gr.Row():
        lens = gr.Radio(
            choices=list(LENS_HINTS.keys()),
            value="Executive",
            label="Audience lens",
            info="Tilts planning and writing without extra prompts from you.",
        )
        depth = gr.Slider(
            2,
            5,
            value=3,
            step=1,
            label="Web searches",
            info="More searches = broader coverage and higher cost.",
        )
    topic = gr.Textbox(label="Research topic", lines=2, placeholder="What should we research?")
    b1 = gr.Button("Step 1 · Generate scoping questions", variant="secondary")
    scope = gr.Markdown(label="Scoping questions")
    answers = gr.Textbox(
        label="Step 2 · Your answers",
        lines=5,
        placeholder="Answer the three questions (short paragraphs are fine).",
    )
    b2 = gr.Button("Step 3 · Run lens-guided research", variant="primary")
    out = gr.Markdown(label="Progress and report")

    b1.click(ui_clarify, inputs=[topic, lens], outputs=scope)
    b2.click(ui_run, inputs=[topic, answers, lens, depth], outputs=out)

demo.launch()
